# ECC Volatility Prediction

It assumes the earnings-call
QA embeddings (with `ticker`/`date` fields already attached by the data-pipeline
notebook) and the KeFVP price/volatility label CSVs are available locally, and it
trains and compares several volatility-prediction models built on top of them.



## Setup & configuration

In [62]:
import os
import pickle
import random
from collections import defaultdict
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_DIR = Path("./data/volatility_embeddings")
RESULTS_DIR = Path("./results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [1, 3, 5, 6, 9, 11, 13, 14, 15, 17, 18, 19
        21, 23, 24, 26, 28, 30, 31, 32, 33, 34, 35,
         39, 40, 42, 43, 44, 45, 46, 48, 50, 53, 59,
         60, 61, 63, 64, 65, 66, 67, 69, 71, 73, 76,
         80, 81, 83, 87, 88, 89, 93, 94, 96, 98]

BATCH_SIZE = 32
QA_EMBED_DIM = 3584
HORIZONS = [3, 7, 15, 30]
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SCHED_PATIENCE = 10
SCHED_FACTOR = 0.5
SCHED_MIN_LR = 1e-6
EARLY_STOP_PATIENCE = 100
NUM_EPOCHS = 100


def get_past_cols(dataset_key: str, split : str = "train"):
    """maec15/maec16 use past_1..past_29; ec uses past_2..past_30."""
    if dataset_key == "ec" and split == "test":
        return [f"past_{i}" for i in range(2, 31)]  # past_2 .. past_30
    return [f"past_{i}" for i in range(1, 30)]  # past_1 .. past_29


target_cols = [f"future_{h}" for h in HORIZONS]

Using device: cuda


## Load price / volatility label CSVs

In [46]:
!wget -q -O maec15_train_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/15/maec15_train_avg_val.csv
!wget -q -O maec15_dev_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/15/maec15_dev_avg_val.csv
!wget -q -O maec15_test_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/15/maec15_test_avg_val.csv

!wget -q -O maec16_train_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/16/maec16_train_avg_val.csv
!wget -q -O maec16_dev_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/16/maec16_dev_avg_val.csv
!wget -q -O maec16_test_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/maec/16/maec16_test_avg_val.csv

!wget -q -O ec_test_avg_val.csv \
    https://raw.githubusercontent.com/hankniu01/KeFVP/main/price_data/test_split_Avg_Series_WITH_LOG.csv

In [47]:
maec15_train_avg_val_df = pd.read_csv("maec15_train_avg_val.csv")
maec15_dev_avg_val_df = pd.read_csv("maec15_dev_avg_val.csv")
maec15_test_avg_val_df = pd.read_csv("maec15_test_avg_val.csv")

maec16_train_avg_val_df = pd.read_csv("maec16_train_avg_val.csv")
maec16_dev_avg_val_df = pd.read_csv("maec16_dev_avg_val.csv")
maec16_test_avg_val_df = pd.read_csv("maec16_test_avg_val.csv")

ec_test_avg_val_df = pd.read_csv("ec_test_avg_val.csv")

maec_combined_train_df = pd.concat(
    [maec15_train_avg_val_df, maec16_train_avg_val_df], ignore_index=True
)
maec_combined_dev_df = pd.concat(
    [maec15_dev_avg_val_df, maec16_dev_avg_val_df], ignore_index=True
)

PRICE_DFS = {
    "maec15": {
        "train": maec15_train_avg_val_df,
        "dev": maec15_dev_avg_val_df,
        "test": maec15_test_avg_val_df,
    },
    "maec16": {
        "train": maec16_train_avg_val_df,
        "dev": maec16_dev_avg_val_df,
        "test": maec16_test_avg_val_df,
    },
    "ec": {
        "train": maec_combined_train_df,
        "dev": maec_combined_dev_df,
        "test": ec_test_avg_val_df,
    },
}


## Load embeddings



In [48]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
import os
import shutil

src_dir = "/content/drive/MyDrive/Volatility-Prediction-Data"
dst_dir = "./data/volatility_embeddings"

os.makedirs(dst_dir, exist_ok=True)

for filename in os.listdir(src_dir):
    if filename.endswith(".pkl"):
        shutil.copy2(
            os.path.join(src_dir, filename),
            os.path.join(dst_dir, filename)
        )

print("Done.")

Done.


In [50]:
MODALITY_SUFFIX = {
    "text_audio_prompt1": "embeddings",
    "text_only": "text_embeddings",
    "text_audio_prompt2": "qa_text_audio_prompt2_embeddings",
    "audio_only": "audio_embeddings",
    "chunk_text_audio": "chunk_text_audio_embeddings",
}

MODALITIES = list(MODALITY_SUFFIX)


def load_embeddings(dataset: str, split: str, modality: str):
    """Load a single embeddings pickle file.

    dataset: "maec15" | "maec16" | "ec"
    split:   "train" | "dev" | "test"  (EC only has "test")
    modality: one of MODALITY_SUFFIX
    """
    suffix = MODALITY_SUFFIX[modality]
    fname = f"kefvp_{split}_{dataset}_{suffix}.pkl"
    path = DATA_DIR / fname
    with open(path, "rb") as f:
        return pickle.load(f)


# embeddings[(dataset, split, modality)] -> list[dict] with "ticker"/"date"/"embedding"
EMBEDDINGS = {}

for _dataset in ("maec15", "maec16"):
    for _split in ("train", "dev", "test"):
        for _modality in MODALITIES:
            EMBEDDINGS[(_dataset, _split, _modality)] = load_embeddings(
                _dataset, _split, _modality
            )

for _modality in MODALITIES:
    EMBEDDINGS[("ec", "test", _modality)] = load_embeddings("ec", "test", _modality)

print(f"Loaded {len(EMBEDDINGS)} embedding files.")

Loaded 35 embedding files.


In [51]:
def build_lookup(embedding_lists):
    """Index a list of embedding-lists by (ticker, date) -> list[np.ndarray]."""
    lookup = defaultdict(list)
    for dataset in embedding_lists:
        for item in dataset:
            key = (item["ticker"], str(item["date"]))
            lookup[key].append(item["embedding"].astype(np.float32))
    return lookup


def get_qa_lists(df, lookup, is_ec: bool = False):
    """For each row of a price/label dataframe, return the list of QA embeddings
     whose (ticker, date) matches that row."""
    result = []

    for _, row in df.iterrows():

        ticker = row["ticker"]

        if is_ec:
            date = f"{int(row['year']):04d}{int(row['month']):02d}{int(row['day']):02d}"
        else:
            date = str(row["time"]).replace("-", "")

        key = (ticker, date)
        qas = lookup.get(key, [])
        if len(qas) == 0:
            continue
        result.append(qas)

    return result




def get_data_and_qa_for(dataset_key: str, split: str, modality: str):
    """
    Return a price/label dataframe and its corresponding QA embeddings,
    aligned row-by-row.
    """

    df = PRICE_DFS[dataset_key][split].copy()

    if dataset_key == "ec":
        if split == "test":
            embedding_lists = [
                EMBEDDINGS[("ec", "test", modality)]
            ]
            is_ec = True
        else:
            embedding_lists = [
                EMBEDDINGS[("maec15", split, modality)],
                EMBEDDINGS[("maec16", split, modality)],
            ]
            is_ec = False
    else:
        embedding_lists = [
            EMBEDDINGS[(dataset_key, split, modality)]
        ]
        is_ec = False

    lookup = build_lookup(embedding_lists)

    rows = []
    qa_lists = []

    for _, row in df.iterrows():

        ticker = row["ticker"]

        if is_ec:
            date = (
                f"{int(row['year']):04d}"
                f"{int(row['month']):02d}"
                f"{int(row['day']):02d}"
            )
        else:
            date = str(row["time"]).replace("-", "")

        qas = lookup.get((ticker, date), [])

        if not qas:
            continue

        rows.append(row)
        qa_lists.append(qas)

    aligned_df = pd.DataFrame(rows).reset_index(drop=True)

    assert len(aligned_df) == len(qa_lists)

    return aligned_df, qa_lists

In [52]:
def get_item_ids(df: pd.DataFrame, is_ec: bool) -> pd.DataFrame:
    """ticker + normalized date string for each row of a test dataframe, aligned
    to df's row order. Mirrors the (ticker, date) key construction used in
    get_data_and_qa_for so ids stay consistent with how QA embeddings were
    looked up for the same rows."""
    if is_ec:
        date = (
            df["year"].astype(int).astype(str).str.zfill(4)
            + df["month"].astype(int).astype(str).str.zfill(2)
            + df["day"].astype(int).astype(str).str.zfill(2)
        )
    else:
        date = df["time"].astype(str).str.replace("-", "", regex=False)
    return pd.DataFrame({"ticker": df["ticker"].values, "date": date.values})

## Shared dataset & model definitions

In [53]:
class VolDataset(Dataset):
    """Volatility-history-only dataset, used for the vol-only baseline."""

    def __init__(self, df, past_cols):
        self.X = df[past_cols].values.astype(np.float32)
        self.y = df[target_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class ECCDataset(Dataset):
    """Volatility history + variable-length list of QA embeddings per example.

    Used for every fusion/QA-only experiment. The "whole chunk" (no-QA) modality
    reuses this same class -- each example's "QA list" there is just a length-1
    list containing the single chunk embedding.
    """

    def __init__(self, df, qa_lists, past_cols):
        self.X = df[past_cols].values.astype(np.float32)
        self.y = df[target_cols].values.astype(np.float32)
        self.qa_lists = qa_lists

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.qa_lists[idx], self.y[idx]


def collate_fn(batch):
    """Pad the variable-length QA lists in a batch to a common length and build a
    boolean mask marking real (non-padding) positions."""
    vols, qas, ys = zip(*batch)

    vols = torch.tensor(np.stack(vols), dtype=torch.float32)
    ys = torch.tensor(np.stack(ys), dtype=torch.float32)

    max_len = max(max(len(x), 1) for x in qas)

    qa_tensor = torch.zeros(len(qas), max_len, QA_EMBED_DIM, dtype=torch.float32)
    mask = torch.zeros(len(qas), max_len, dtype=torch.bool)

    for i, qa_list in enumerate(qas):
        if len(qa_list) == 0:
            continue
        arr = np.stack(qa_list)
        qa_tensor[i, : len(arr)] = torch.tensor(arr)
        mask[i, : len(arr)] = True

    return vols, qa_tensor, mask, ys

In [54]:
class AttentionPooling(nn.Module):
    """Masked attention pooling over a padded sequence of QA embeddings."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)

        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-8)

        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled

In [55]:
def run_packed_bilstm(lstm, x, mask):
    """Run `lstm` over a padded, batch-first sequence via pack_padded_sequence.

    Without packing, a bidirectional LSTM's backward pass starts at the padded
    tail and only reaches real tokens after stepping through the pad positions
    first, so its hidden state at every real position is contaminated by a
    batch-dependent number of leading pad steps. Packing removes those pad
    steps from the recurrence entirely; masking in AttentionPooling alone does
    not fix this since it only zeroes pooling weights after the LSTM has run.
    """
    lengths = mask.sum(dim=1).clamp(min=1).to("cpu")
    packed = nn.utils.rnn.pack_padded_sequence(
        x, lengths, batch_first=True, enforce_sorted=False
    )
    packed_out, _ = lstm(packed)
    out, _ = nn.utils.rnn.pad_packed_sequence(
        packed_out, batch_first=True, total_length=x.size(1)
    )
    return out

In [56]:
class VolOnlyModel(nn.Module):
    """Volatility-history-only baseline: 2-layer LSTM over `past_cols`,

    This is the frozen baseline whose `vol_lstm` + `head` get reused (frozen) by
    `VolECCResidualGatedModel`.
    """

    def __init__(self):
        super().__init__()

        self.vol_lstm = nn.LSTM(
            input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
        )

        self.head = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, vol):
        vol = vol.unsqueeze(-1)  # [B, 28] -> [B, 28, 1]
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]  # [B, 64]
        return self.head(vol_repr)

In [57]:
class QAOnlyModel(nn.Module):
    """QA-embeddings-only model: no volatility branch at all."""

    def __init__(self):
        super().__init__()

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, qa, mask):
        qa = self.qa_proj(qa)
        qa_out = run_packed_bilstm(self.qa_lstm, qa, mask)
        qa_repr = self.attention_pool(qa_out, mask)
        return self.head(qa_repr)

In [58]:
class VolECCConcatModel(nn.Module):
    """Concat fusion: vol_lstm (trained from scratch) || (qa_proj -> BiLSTM ->
    AttentionPooling), concatenated and passed through an MLP head."""

    def __init__(self):
        super().__init__()

        self.vol_lstm = nn.LSTM(
            input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
        )

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.head = nn.Sequential(
            nn.Linear(64 + 256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, vol, qa, mask):
        vol = vol.unsqueeze(-1)
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]

        qa = self.qa_proj(qa)
        qa_out = run_packed_bilstm(self.qa_lstm, qa, mask)
        qa_repr = self.attention_pool(qa_out, mask)

        fused = torch.cat([vol_repr, qa_repr], dim=-1)
        out = self.head(fused)
        return out

In [ ]:
# class VolECCGatedNoBiLSTMModel(nn.Module):
#     """Gated fusion, no BiLSTM on the QA branch: attention pooling is applied
#     directly to `qa_proj`'s output, and a learned gate scales the pooled QA
#     representation before concatenation with the volatility representation."""

#     def __init__(self):
#         super().__init__()

#         self.vol_lstm = nn.LSTM(
#             input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
#         )

#         self.qa_proj = nn.Sequential(
#             nn.Linear(QA_EMBED_DIM, 512),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(512, 256),
#         )

#         self.attention_pool = AttentionPooling(hidden_dim=256)

#         self.qa_gate = nn.Sequential(
#             nn.Linear(64 + 256, 128),
#             nn.ReLU(),
#             nn.Linear(128, 256),
#             nn.Sigmoid(),
#         )

#         self.head = nn.Sequential(
#             nn.Linear(64 + 256, 128),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(128, 64),
#             nn.ReLU(),
#             nn.Linear(64, 4),
#         )

#     def forward(self, vol, qa, mask):
#         vol = vol.unsqueeze(-1)
#         _, (h, _) = self.vol_lstm(vol)
#         vol_repr = h[-1]

#         qa = self.qa_proj(qa)
#         qa_repr = self.attention_pool(qa, mask)  # attention directly on qa_proj output

#         gate = self.qa_gate(torch.cat([vol_repr, qa_repr], dim=-1))
#         qa_repr = gate * qa_repr

#         fused = torch.cat([vol_repr, qa_repr], dim=-1)
#         out = self.head(fused)
#         return out

In [59]:
class VolECCResidualGatedModel(nn.Module):
    """Residual/frozen fusion: a frozen, pretrained `VolOnlyModel` (`vol_lstm` +
    `head`, both `requires_grad=False`) predicts a base volatility forecast; a QA
    branch (proj -> BiLSTM -> AttentionPooling -> qa_reduce) predicts a delta on
    top of it. `delta_head`'s final layer is zero-initialized so the model starts
    out numerically identical to the frozen vol-only baseline and has to learn to
    deviate from it.
    """

    def __init__(self, vol_model: "VolOnlyModel"):
        super().__init__()

        # Reuse the pretrained volatility branch, frozen.
        self.vol_lstm = vol_model.vol_lstm
        self.vol_head = vol_model.head
        for p in self.vol_lstm.parameters():
            p.requires_grad = False
        for p in self.vol_head.parameters():
            p.requires_grad = False

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.qa_reduce = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
        )

        self.delta_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 4),
        )
        nn.init.zeros_(self.delta_head[-1].weight)
        nn.init.zeros_(self.delta_head[-1].bias)

    def train(self, mode: bool = True):
        # requires_grad=False above only blocks weight updates; it doesn't stop
        # dropout inside the frozen branch from being re-enabled by model.train().
        # Force it back to eval() every time so the frozen baseline stays
        # deterministic during training, not just during evaluation.
        super().train(mode)
        self.vol_lstm.eval()
        self.vol_head.eval()
        return self

    def forward(self, vol, qa, mask):
        vol = vol.unsqueeze(-1)
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]

        qa = self.qa_proj(qa)
        qa_out = run_packed_bilstm(self.qa_lstm, qa, mask)
        qa_repr = self.attention_pool(qa_out, mask)
        qa_repr = self.qa_reduce(qa_repr)

        vol_pred = self.vol_head(vol_repr)
        delta_input = torch.cat([vol_repr, qa_repr], dim=-1)
        delta = self.delta_head(delta_input)

        out = vol_pred + delta
        return out, vol_pred, delta


FUSION_MODEL_CLASSES = {
    "concat": VolECCConcatModel,
    "residual_gated": VolECCResidualGatedModel,
    "qa_only": QAOnlyModel,
}

## Vol-only baseline pretraining

Trains `VolOnlyModel` per seed for each dataset key that a fusion experiment will
later need frozen weights from.

In [60]:
def train_vol_only_baseline(dataset_key: str, seeds=SEEDS):
    """Train VolOnlyModel once per seed for one dataset key, returning
    {seed: state_dict}, a per-seed results dataframe, and a long-format
    per-test-item loss dataframe (one row per seed x test item x horizon)."""

    train_df = PRICE_DFS[dataset_key]["train"]
    dev_df = PRICE_DFS[dataset_key]["dev"]
    test_df = PRICE_DFS[dataset_key]["test"]
    past_cols = get_past_cols(dataset_key)
    if dataset_key == 'ec':
      test_past_cols = get_past_cols(dataset_key, 'test')
    else:
      test_past_cols = past_cols

    item_ids = get_item_ids(test_df, is_ec=(dataset_key == "ec"))

    best_states = {}
    all_results = []
    all_item_results = []

    for seed in seeds:
        print(f"\n[vol_only:{dataset_key}] seed {seed}")

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_loader = DataLoader(VolDataset(train_df, past_cols), batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(VolDataset(dev_df, past_cols), batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(VolDataset(test_df, test_past_cols), batch_size=BATCH_SIZE, shuffle=False)

        model = VolOnlyModel().to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )

        best_val = float("inf")
        best_state = None
        counter = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for vol, y in train_loader:
                vol, y = vol.to(device), y.to(device)
                optimizer.zero_grad()
                pred = model(vol)
                loss = criterion(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vol, y in dev_loader:
                    vol, y = vol.to(device), y.to(device)
                    val_loss += criterion(model(vol), y).item()
            val_loss /= len(dev_loader)

            if val_loss < best_val:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= EARLY_STOP_PATIENCE:
                    break

        best_states[seed] = best_state
        model.load_state_dict(best_state)

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for vol, y in test_loader:
                vol = vol.to(device)
                preds.append(model(vol).cpu().numpy())
                targets.append(y.numpy())
        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        mses = [
            mean_squared_error(targets[:, i], preds[:, i]) for i in range(len(HORIZONS))
        ]
        avg_mse = float(np.mean(mses))
        print(f"  avg_mse={avg_mse:.6f}")

        result = {"seed": seed, "avg_mse": avg_mse}
        for i, hrz in enumerate(HORIZONS):
            result[f"mse_{hrz}"] = mses[i]
        all_results.append(result)

        for i, hrz in enumerate(HORIZONS):
            all_item_results.append(pd.DataFrame({
                "seed": seed,
                "dataset": dataset_key,
                "modality": "vol_only",
                "fusion_type": "n/a",
                "ticker": item_ids["ticker"],
                "date": item_ids["date"],
                "horizon": hrz,
                "pred": preds[:, i],
                "target": targets[:, i],
                "sq_err": (preds[:, i] - targets[:, i]) ** 2,
            }))

        torch.save(best_state, RESULTS_DIR / f"vol_only_state_{dataset_key}_{seed}.pt")

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(RESULTS_DIR / f"results_{dataset_key}_vol_only.csv", index=False)

    item_results_df = pd.concat(all_item_results, ignore_index=True)

    return best_states, results_df, item_results_df


VOL_ONLY_DATASET_KEYS = ["maec16"]

best_models = {}
vol_only_results = {}
vol_only_item_results = {}

for _dataset_key in VOL_ONLY_DATASET_KEYS:
    _states, _results, _item_results = train_vol_only_baseline(_dataset_key)
    best_models[_dataset_key] = _states
    vol_only_results[_dataset_key] = _results
    vol_only_item_results[_dataset_key] = _item_results

print({k: v["avg_mse"].mean() for k, v in vol_only_results.items()})


[vol_only:maec16] seed 18
  avg_mse=0.224568

[vol_only:maec16] seed 19
  avg_mse=0.229925
{'maec16': np.float64(0.22724664211273193)}


## Generic experiment runner

In [64]:
def run_experiment(
    name: str,
    dataset_key: str,
    modality: str,
    fusion_type: str,
    train_qas,
    dev_qas,
    test_qas,
    train_df,
    dev_df,
    test_df,
    needs_vol_pretrain: bool = True,
    seeds=SEEDS,
):
    """Train+evaluate one (dataset, modality, fusion_type) combination across all
    seeds, returning a per-seed results dataframe and a long-format per-test-item
    loss dataframe (one row per seed x test item x horizon); the results
    dataframe is also written to RESULTS_DIR.

    fusion_type: "concat"   | "residual_gated" | "qa_only"
    needs_vol_pretrain: True for "residual_gated" (loads the frozen VolOnlyModel
        weights trained in the previous section); ignored otherwise.
    """
    model_cls = FUSION_MODEL_CLASSES[fusion_type]
    past_cols = get_past_cols(dataset_key)
    if dataset_key == 'ec':
      test_past_cols = get_past_cols(dataset_key, 'test')
    else:
      test_past_cols = past_cols

    all_results = []
    all_item_results = []

    item_ids = get_item_ids(test_df, is_ec=(dataset_key == "ec"))

    for seed in seeds:
        print(f"\n[{name}] seed {seed}")

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_loader = DataLoader(
            ECCDataset(train_df, train_qas, past_cols), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
        )
        dev_loader = DataLoader(
            ECCDataset(dev_df, dev_qas, past_cols), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )
        test_loader = DataLoader(
            ECCDataset(test_df, test_qas, test_past_cols), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )

        if fusion_type == "residual_gated" and needs_vol_pretrain:
            vol_model = VolOnlyModel().to(device)
            vol_model.load_state_dict(best_models[dataset_key][seed])
            model = model_cls(vol_model).to(device)
        else:
            model = model_cls().to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=SCHED_FACTOR,
            patience=SCHED_PATIENCE,
            threshold=1e-4,
            min_lr=SCHED_MIN_LR,
        )

        best_val = float("inf")
        best_state = None
        counter = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for vol, qa, mask, y in train_loader:
                vol, qa, mask, y = vol.to(device), qa.to(device), mask.to(device), y.to(device)
                optimizer.zero_grad()

                if fusion_type == "residual_gated":
                    pred, _, _ = model(vol, qa, mask)
                elif fusion_type == "qa_only":
                    pred = model(qa, mask)
                else:
                    pred = model(vol, qa, mask)

                loss = criterion(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vol, qa, mask, y in dev_loader:
                    vol, qa, mask, y = vol.to(device), qa.to(device), mask.to(device), y.to(device)
                    if fusion_type == "residual_gated":
                        pred, _, _ = model(vol, qa, mask)
                    elif fusion_type == "qa_only":
                        pred = model(qa, mask)
                    else:
                        pred = model(vol, qa, mask)
                    val_loss += criterion(pred, y).item()
            val_loss /= len(dev_loader)
            scheduler.step(val_loss)

            if val_loss < best_val:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= EARLY_STOP_PATIENCE:
                    break

        model.load_state_dict(best_state)
        model.eval()

        preds, targets = [], []
        with torch.no_grad():
            for vol, qa, mask, y in test_loader:
                vol, qa, mask = vol.to(device), qa.to(device), mask.to(device)
                if fusion_type == "residual_gated":
                    pred, _, _ = model(vol, qa, mask)
                elif fusion_type == "qa_only":
                    pred = model(qa, mask)
                else:
                    pred = model(vol, qa, mask)
                preds.append(pred.cpu().numpy())
                targets.append(y.numpy())

        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        mses = [
            mean_squared_error(targets[:, i], preds[:, i]) for i in range(len(HORIZONS))
        ]
        avg_mse = float(np.mean(mses))
        print(f"  avg_mse={avg_mse:.6f}")

        result = {
            "seed": seed,
            "dataset": dataset_key,
            "modality": modality,
            "fusion_type": fusion_type,
            "avg_mse": avg_mse,
        }
        for i, hrz in enumerate(HORIZONS):
            result[f"mse_{hrz}"] = mses[i]
        all_results.append(result)

        for i, hrz in enumerate(HORIZONS):
            all_item_results.append(pd.DataFrame({
                "seed": seed,
                "dataset": dataset_key,
                "modality": modality,
                "fusion_type": fusion_type,
                "ticker": item_ids["ticker"],
                "date": item_ids["date"],
                "horizon": hrz,
                "pred": preds[:, i],
                "target": targets[:, i],
                "sq_err": (preds[:, i] - targets[:, i]) ** 2,
            }))

    results_df = pd.DataFrame(all_results)


    if fusion_type == "residual_gated":
        csv_name = f"results_{dataset_key}_{modality}.csv"
    else:
        csv_name = f"results_{dataset_key}_{modality}_{fusion_type}.csv"

    results_df.to_csv(RESULTS_DIR / csv_name, index=False)

    item_results_df = pd.concat(all_item_results, ignore_index=True)

    return results_df, item_results_df

## Experiment configuration table




In [23]:
MODALITIES = ["chunk_text_audio","audio_only","text_only", "text_audio_prompt1", 'text_audio_prompt2']



DATASETS = ["maec16"]
FUSION_TYPES = ["residual_gated"]


PROMPT2 = "text_audio_prompt2"


EXPERIMENT_CONFIG = []

for modality in MODALITIES:
    for dataset in DATASETS:
        for fusion_type in FUSION_TYPES:
            EXPERIMENT_CONFIG.append({
                "dataset": dataset,
                "modality": modality,
                "fusion_type": fusion_type,
            })

    if modality == PROMPT2:
        for dataset in DATASETS:
            EXPERIMENT_CONFIG.append({
                "dataset": dataset,
                "modality": modality,
                "fusion_type": "qa_only",
            })
            EXPERIMENT_CONFIG.append({
                "dataset": dataset,
                "modality": modality,
                "fusion_type": "concat",
            })


experiment_config_df = pd.DataFrame(EXPERIMENT_CONFIG)
print(f"{len(experiment_config_df)} experiment rows")
experiment_config_df

7 experiment rows


,dataset,modality,fusion_type
0,maec16,chunk_text_audio,residual_gated
1,maec16,audio_only,residual_gated
2,maec16,text_only,residual_gated
3,maec16,text_audio_prompt1,residual_gated
4,maec16,text_audio_prompt2,residual_gated
5,maec16,text_audio_prompt2,qa_only
6,maec16,text_audio_prompt2,concat


## Run all experiments

Iterates the config table above, loads the right QA lists for each
`(dataset, modality)` pair once, and calls `run_experiment` for each row. Results
from every row are concatenated into one `all_results_df`; each row also writes
its own CSV via `run_experiment`.

In [24]:
all_results = []
all_item_results = []
_qa_cache = {}

for _, cfg in experiment_config_df.iterrows():
    dataset_key = cfg["dataset"]
    modality = cfg["modality"]
    fusion_type = cfg["fusion_type"]
    name = f"{dataset_key}_{modality}_{fusion_type}"

    cache_key = (dataset_key, modality)
    if cache_key not in _qa_cache:
        _qa_cache[cache_key] = {
            "train": get_data_and_qa_for(dataset_key, "train", modality),
            "dev": get_data_and_qa_for(dataset_key, "dev", modality),
            "test": get_data_and_qa_for(dataset_key, "test", modality),
        }
    train_df, train_qas = _qa_cache[cache_key]["train"]
    dev_df, dev_qas = _qa_cache[cache_key]["dev"]
    test_df, test_qas = _qa_cache[cache_key]["test"]

    result_df, item_result_df = run_experiment(
        name=name,
        dataset_key=dataset_key,
        modality=modality,
        fusion_type=fusion_type,
        train_qas=train_qas,
        dev_qas=dev_qas,
        test_qas=test_qas,
        train_df=train_df,
        dev_df=dev_df,
        test_df=test_df,
        needs_vol_pretrain=(fusion_type == "residual_gated"),
    )
    all_results.append(result_df)
    all_item_results.append(item_result_df)

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(RESULTS_DIR / "all_results.csv", index=False)

all_item_losses_df = pd.concat(
    [*vol_only_item_results.values(), *all_item_results],
    ignore_index=True,
)
all_item_losses_df.to_csv(RESULTS_DIR / "item_losses.csv", index=False)



[maec16_chunk_text_audio_residual_gated] seed 19
  avg_mse=0.212762

[maec16_audio_only_residual_gated] seed 19
  avg_mse=0.237514

[maec16_text_only_residual_gated] seed 19
  avg_mse=0.221544

[maec16_text_audio_prompt1_residual_gated] seed 19
  avg_mse=0.232393

[maec16_text_audio_prompt2_residual_gated] seed 19
  avg_mse=0.213233

[maec16_text_audio_prompt2_qa_only] seed 19
  avg_mse=0.272325

[maec16_text_audio_prompt2_concat] seed 19
  avg_mse=0.307167


,seed,dataset,modality,fusion_type,avg_mse,mse_3,mse_7,mse_15,mse_30
0,19,maec16,chunk_text_audio,residual_gated,0.212762,0.316325,0.179344,0.212225,0.143153
1,19,maec16,audio_only,residual_gated,0.237514,0.347814,0.207485,0.238348,0.156408
2,19,maec16,text_only,residual_gated,0.221544,0.336620,0.186864,0.218478,0.144216
3,19,maec16,text_audio_prompt1,residual_gated,0.232393,0.346163,0.198299,0.233178,0.151932
4,19,maec16,text_audio_prompt2,residual_gated,0.213233,0.319960,0.180006,0.212371,0.140595


In [ ]:
all_item_losses_df = pd.concat(
    [*vol_only_item_results.values(), *all_item_results],
    ignore_index=True,
)
all_item_losses_df.to_csv(RESULTS_DIR / "item_losses.csv", index=False)

In [ ]:
all_item_losses_df

,seed,dataset,modality,fusion_type,ticker,date,horizon,pred,target,sq_err
0,1,ec,vol_only,n/a,LMT,20171024,3,-4.703795,-5.038498,0.112026
1,1,ec,vol_only,n/a,GM,20171024,3,-4.635042,-4.333446,0.090961
2,1,ec,vol_only,n/a,ILMN,20171024,3,-4.658343,-3.932037,0.527520
3,1,ec,vol_only,n/a,TXN,20171024,3,-4.682132,-4.784488,0.010477
4,1,ec,vol_only,n/a,UTX,20171024,3,-4.698135,-4.976168,0.077302
...,...,...,...,...,...,...,...,...,...,...
200155,19,ec,text_audio_prompt2,concat,DG,20171207,30,-4.854618,-4.535144,0.102063
200156,19,ec,text_audio_prompt2,concat,ORCL,20171214,30,-4.614616,-4.415145,0.039788
200157,19,ec,text_audio_prompt2,concat,DRI,20171219,30,-4.536293,-4.595548,0.003511
200158,19,ec,text_audio_prompt2,concat,GIS,20171220,30,-4.602782,-4.449197,0.023588


## Results aggregation & comparison

In [26]:
mse_cols = [f"mse_{h}" for h in HORIZONS]

summary_df = (
    all_results_df
    .groupby(["dataset", "modality", "fusion_type"])[mse_cols + ["avg_mse"]]
    .agg(["mean", "std"])
)
summary_df

mse_3         mse_7      \
                                               mean std      mean std   
dataset modality           fusion_type                                  
maec16  audio_only         residual_gated  0.347814 NaN  0.207485 NaN   
        chunk_text_audio   residual_gated  0.316325 NaN  0.179344 NaN   
        text_audio_prompt1 residual_gated  0.346163 NaN  0.198299 NaN   
        text_audio_prompt2 concat          0.384726 NaN  0.257902 NaN   
                           qa_only         0.404338 NaN  0.252056 NaN   
                           residual_gated  0.319960 NaN  0.180006 NaN   
        text_only          residual_gated  0.336620 NaN  0.186864 NaN   

                                             mse_15        mse_30      \
                                               mean std      mean std   
dataset modality           fusion_type                                  
maec16  audio_only         residual_gated  0.238348 NaN  0.156408 NaN   
        chunk_text_audio   residual_gated  0.212225 NaN  0.143153 NaN   
        text_audio_prompt1 residual_gated  0.233178 NaN  0.151932 NaN   
        text_audio_prompt2 concat          0.326602 NaN  0.259437 NaN   
                           qa_only         0.232906 NaN  0.200002 NaN   
                           residual_gated  0.212371 NaN  0.140595 NaN   
        text_only          residual_gated  0.218478 NaN  0.144216 NaN   

                                            avg_mse      
                                               mean std  
dataset modality           fusion_type                   
maec16  audio_only         residual_gated  0.237514 NaN  
        chunk_text_audio   residual_gated  0.212762 NaN  
        text_audio_prompt1 residual_gated  0.232393 NaN  
        text_audio_prompt2 concat          0.307167 NaN  
                           qa_only         0.272325 NaN  
                           residual_gated  0.213233 NaN  
        text_only          residual_gated  0.221544 NaN